In [1]:
import warnings
import logging

root_logger = logging.getLogger()
root_logger.setLevel(logging.ERROR)
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import wave
import struct
from scipy.signal import find_peaks, hilbert

# Generate a sample sine wave .Wav file
def generate_sine_wave(filename, freq=440, duration=2, sample_rate=44100, amplitude=0.5):
    t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)
    data = amplitude * np.sin(2 * np.pi * freq * t)
    # Convert to 16-bit PCM
    pcm_data = np.int16(data * 32767)
    with wave.open(filename, 'w') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sample_rate)
        wf.writeframes(pcm_data.tobytes())

# Analyze .Wav file: extract frequency and amplitude envelope
def analyze_wav(filename):
    with wave.open(filename, 'r') as wf:
        n_frames = wf.getnframes()
        sample_rate = wf.getframerate()
        audio = wf.readframes(n_frames)
        data = np.frombuffer(audio, dtype=np.int16)
        data = data / 32767.0  # Normalize

    # Frequency analysis (FFT)
    fft = np.fft.fft(data)
    freqs = np.fft.fftfreq(len(fft), 1/sample_rate)
    idx = np.argmax(np.abs(fft[:len(fft)//2]))
    peak_freq = abs(freqs[idx])

    # Amplitude envelope (Hilbert transform)
    analytic_signal = hilbert(data)
    amplitude_envelope = np.abs(analytic_signal)
    max_amp = np.max(amplitude_envelope)
    min_amp = np.min(amplitude_envelope)

    return {
        "peak_frequency": peak_freq,
        "max_amplitude": max_amp,
        "min_amplitude": min_amp,
        "duration": len(data) / sample_rate,
        "sample_rate": sample_rate
    }

# Write synthesizer data to text file
def write_synth_data(data, out_file):
    with open(out_file, 'w') as f:
        for k, v in data.items():
            f.write(f"{k}: {v}\n")

# Main execution
wav_file = "sample_sine.wav"
generate_sine_wave(wav_file)
synth_data = analyze_wav(wav_file)
write_synth_data(synth_data, "synth_data.txt")